# 01 — Tabular EDA on CDFG handcrafted features


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ll_hls4ml.config import load_config
from ll_hls4ml.features.tabular import build_feature_dataframe
from ll_hls4ml.viz.eda import (
    between_class_variance_ratio,
    compute_pca,
    plot_lda_weights,
    plot_tsne,
    plot_tsne_continuous,
)
from sklearn.preprocessing import StandardScaler

import pandas as pd


/home/brend/anaconda3/envs/pipeline-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import json
split = "test"
kernel_type = "2_20"
with open(f"../../data/labels/wa-hls4ml/{split}/{split}_{kernel_type}_merged.json", "r") as f:
    data = json.load(f)

hls_res_label = 0
res_label = 0
lat_label = 0
for row in data:
    if row["hls_resource_report"]:
        hls_res_label += 1
    if row["resource_report"]:
        res_label += 1
    if row["latency_report"]:
        lat_label += 1

print(f"Proportion of projects with HLS resource results: {hls_res_label / len(data):2.2%}")
print(f"Proportion of projects with latency results: {lat_label / len(data):2.2%}")
print(f"Proportion of projects with resource results: {res_label / len(data):2.2%}")


Proportion of projects with HLS resource results: 0.00%
Proportion of projects with latency results: 100.00%
Proportion of projects with resource results: 100.00%


In [4]:
cfg = load_config()

kernel_subset = [
    "2layer", "exemplar", "conv1d", "conv2d",
    "dense_latency", "dense_resource",
]
max_archives = 2

df = build_feature_dataframe(cfg.graph_dir, kernel_subset="conv1d", max_archives=max_archives)
df = df.fillna(0.0)
df.sample(20)


Parsing graph files for kernel subset 'conv1d':   7%|▋         | 7/100 [00:10<02:19,  1.50s/it]


KeyboardInterrupt: 

In [ ]:
exclude_cols = [
    "kernel_type", "bram", "cycles_max", "cycles_min", "dsp",
    "estimated_clock", "ff", "interval_max", "interval_min",
    "lut", "target_clock", "uram",
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].values
y = df[exclude_cols]
X_scaled = StandardScaler().fit_transform(X)


In [ ]:
df.groupby("kernel_type").agg(["mean", "std", "min", "max"])


In [ ]:
embedding_pca, pca, energy_df = compute_pca(X_scaled, pca_components=2, feature_names=feature_cols)
plot_tsne(X_scaled, y["kernel_type"].values)
plot_tsne_continuous(X_scaled, df, label_name="lut", log_y=True)


In [ ]:
bcvr_df = between_class_variance_ratio(X_scaled, y["kernel_type"].values, feature_cols)
bcvr_df.head(20)


In [ ]:
lda_weights = plot_lda_weights(X_scaled, y["kernel_type"].values, feature_cols)
lda_weights.head(20)


In [ ]:
# Optional export
df.groupby("kernel_type").agg(["mean", "std", "min", "max"]).to_csv(cfg.exports_dir / "kernel_stats.csv")
